[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junseo25/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026_10_16_Jun_Seo_Lee__%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb)

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

             sku          product_name  ...           supplier package_size
0          PT-12  pop-tarts strawberry  ...  Johnson & Johnson       4-pack
1     SKU#459812   Strawberry Pop-Tart  ...           Roku Inc          4pk
2  POPTART-STRAW  Pop-Tarts Strawberry  ...           Sara Lee       6 pack
3  PT_STRAWBERRY  pop-tarts strawberry  ...         Campbell's       4 pack
4         WTR-24    BOTTLED WATER 24PK  ...           Sara Lee       1 pack

[5 rows x 7 columns]

### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# Make a Boolean mask. Lowercasing first makes the search case-insensitive.
pop_mask = cat['product_name'].str.lower().str.contains('pop', na=False)

# .copy() gives us an independent table that is safe to work with later.
pop_rows = cat.loc[pop_mask].copy()

print('Number of Pop-Tart rows:', len(pop_rows))
pop_rows[['sku', 'product_name', 'category']]

Number of Pop-Tart rows: 4


             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# Search only within the Pop-Tart rows found in Q1.
strawberry_mask = (
    pop_rows['product_name']
    .str.lower()
    .str.contains('straw', na=False)
)

strawberry_pop_tarts = pop_rows.loc[strawberry_mask].copy()
other_pop_tarts = pop_rows.loc[~strawberry_mask].copy()

print('Strawberry Pop-Tart variants:')
print(strawberry_pop_tarts[['sku', 'product_name']].to_string(index=False))
print('\nPop-Tart rows that are not strawberry:')
if other_pop_tarts.empty:
    print('None in this catalog')
else:
    print(other_pop_tarts[['sku', 'product_name']].to_string(index=False))

Strawberry Pop-Tart variants:
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Pop-Tart rows that are not strawberry:
None in this catalog


**Pop-Tart rows that are a different product:** None appear in this version of the catalog. If another flavor such as blueberry appeared, it would remain separate because a different flavor is a different product for ordering and inventory purposes.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# All four rows describe the same strawberry Pop-Tart product.
# Keep the already-readable POPTART-STRAW value as the canonical SKU.
STRAWBERRY_SKU_MAP = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

mapped = cat.copy()

# fillna(mapped['sku']) is essential: unmapped products keep their original SKU.
mapped['canonical_sku'] = (
    mapped['sku']
    .map(STRAWBERRY_SKU_MAP)
    .fillna(mapped['sku'])
)

mapped.loc[pop_mask, ['sku', 'product_name', 'canonical_sku']]

             sku          product_name  canonical_sku
0          PT-12  pop-tarts strawberry  POPTART-STRAW
1     SKU#459812   Strawberry Pop-Tart  POPTART-STRAW
2  POPTART-STRAW  Pop-Tarts Strawberry  POPTART-STRAW
3  PT_STRAWBERRY  pop-tarts strawberry  POPTART-STRAW

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print('Category counts before cleanup:')
print(mapped['category'].value_counts(dropna=False).sort_index())

# First do the mechanical cleanup requested in the question.
mapped['category_clean'] = mapped['category'].str.strip().str.title()

print('\nAfter strip and title-case:')
print(mapped['category_clean'].value_counts(dropna=False).sort_index())

# Formatting cannot recognize synonyms, so document the business rules explicitly.
CATEGORY_MAP = {
    'Breakfast Foods': 'Breakfast',
    'Alcoholic Beverages': 'Alcohol',
    'Baby Products': 'Baby',
    'Emergency Supplies': 'Emergency',
    'Groceries': 'Grocery',
}
mapped['category_clean'] = mapped['category_clean'].replace(CATEGORY_MAP)

print('\nAfter consolidating labels that mean the same thing:')
print(mapped['category_clean'].value_counts(dropna=False).sort_index())

Category counts before cleanup:
category
ALCOHOL                1
Alcoholic Beverages    1
Baby                   2
Baby Products          1
Breakfast              2
Breakfast Foods        1
ELECTRONICS            2
EMERGENCY              3
ENTERTAINMENT          3
Electronics            1
Emergency              5
Emergency Supplies     3
GROCERY                3
Groceries              1
Grocery                1
HOUSEHOLD              1
Household              1
Snacks                 1
baby                   1
breakfast              1
electronics            1
emergency              1
entertainment          2
grocery                1
health                 2
household              2
snacks                 1
Name: count, dtype: int64

After strip and title-case:
category_clean
Alcohol                1
Alcoholic Beverages    1
Baby                   3
Baby Products          1
Breakfast              3
Breakfast Foods        1
Electronics            4
Emergency              9
Emergency Supp

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
def make_name_key(name):
    # By the time this function runs, punctuation has been changed to spaces.
    # Sorting makes 'Board Game Monopoly' match 'Monopoly Board Game'.
    return ' '.join(sorted(name.split()))

candidate_rows = mapped.copy()
candidate_rows['name_key'] = (
    candidate_rows['product_name']
    .str.lower()
    .str.replace(r'[^a-z0-9]+', ' ', regex=True)
    .str.strip()
    .apply(make_name_key)
)

# Group matching keys, then keep only groups containing multiple SKUs.
candidate_groups = (
    candidate_rows
    .groupby('name_key')
    .agg(
        sku_count=('sku', 'nunique'),
        skus=('sku', lambda values: ', '.join(sorted(values))),
        names=('product_name', lambda values: ' | '.join(sorted(set(values)))),
    )
    .reset_index()
)

duplicate_candidates = candidate_groups.loc[
    candidate_groups['sku_count'] > 1
].sort_values(['sku_count', 'name_key'], ascending=[False, True])

duplicate_candidates

                name_key  ...                                        names
12   board game monopoly  ...  Board Game - Monopoly | Monopoly Board Game
27  pop strawberry tarts  ...  Pop-Tarts Strawberry | pop-tarts strawberry
2     24 beer light pack  ...                         Light Beer (24-pack)
8    4 aa batteries pack  ...                          AA Batteries 4-Pack
10         aid first kit  ...                First Aid Kit | First-Aid Kit
13           bread white  ...                                  WHITE BREAD
14         butter peanut  ...                PEANUT BUTTER | Peanut Butter
15     candles emergency  ...      Candles (Emergency) | Emergency Candles
16           canned soup  ...                  Canned Soup | Soup - Canned
17         cards playing  ...                PLAYING CARDS | Playing Cards
20    chips pack variety  ...    Chips Variety Pack | Variety Pack - Chips
22       duty heavy tarp  ...          Heavy Duty Tarp | TARP - HEAVY DUTY
23        flashlight led 

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# Consolidation should rename values, not remove records or create missing IDs.
assert mapped['canonical_sku'].notna().all(), 'Canonical SKU contains nulls'
assert len(mapped) == len(cat), 'The mapping changed the row count'

print('Safety checks passed.')
print('Rows before mapping:', len(cat))
print('Rows after mapping: ', len(mapped))
print('Null canonical SKUs:', mapped['canonical_sku'].isna().sum())

Safety checks passed.
Rows before mapping: 45
Rows after mapping:  45
Null canonical SKUs: 0


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Two clearly fragmented families beyond Pop-Tarts are:

- **AA batteries:** `BATT-100`, `BAT-AA4`, and `BATT_AA_4PK` all describe a four-pack of AA batteries. I would map all three to one canonical battery SKU after confirming that brand and specifications are not stored elsewhere.
- **Monopoly:** `BG-MONO`, `MONOPOLY-01`, and `GAME-MONO` all describe the Monopoly board game. I would map them to one canonical Monopoly SKU.

The programmatic list in Q5 identifies these as candidates rather than automatically merging them. A matching name is useful evidence, but it is not proof that two sellable items are identical.

**b)** Consolidation becomes a business decision when the available fields cannot prove that two records represent the same sellable item. For example, `STREAM-01` is **Roku Streaming Stick**, while `ROKU-STD` is only **Streaming Device**. They might be the same product, but they could also be different Roku models with different features or prices. I would ask the merchandising/category manager or the product-master-data owner and check vendor identifiers, model numbers, and package details before merging them.